# Gait Analysis Pipeline

Orchestration notebook for EMG/ROM analysis from SWalker platform trials.

All signal processing and metric computation is delegated to the `gait_analysis` package.
This notebook handles configuration, runs the pipeline, and renders visualizations.

**Data layout expected:**
```
data/
├── EMG/   ← semicolon-delimited CSV files, one per trial
└── ROM/   ← Excel XLSX files, one per trial
```
File naming convention: `<patient>_<velocity>_<weight_support>.csv / .xlsx`  
where velocity ∈ {baja, medi, alta} and weight_support ∈ {0, 25, 50}.

In [ ]:
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# Make the package importable when running from the notebooks/ directory
sys.path.insert(0, str(Path(".").resolve().parent))

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(name)s | %(message)s",
)

from gait_analysis.config import DEFAULT_EMG_DIR, DEFAULT_OUTPUT_DIR, DEFAULT_ROM_DIR  # noqa: E402

EMG_DIR = DEFAULT_EMG_DIR
ROM_DIR = DEFAULT_ROM_DIR
OUTPUT_DIR = DEFAULT_OUTPUT_DIR

print(f"EMG dir : {EMG_DIR}")
print(f"ROM dir : {ROM_DIR}")
print(f"Output  : {OUTPUT_DIR}")

## Load data

In [ ]:
from gait_analysis.data.loader import load_emg_files, load_rom_files

emg_data = load_emg_files(EMG_DIR)
rom_data = load_rom_files(ROM_DIR)

print(f"EMG files loaded : {len(emg_data)}")
print(f"ROM files loaded : {len(rom_data)}")

# Preview one trial
sample_key = next(iter(emg_data))
print(f"\nSample key: {sample_key}")
emg_data[sample_key].head()

## Spot-check: signal preprocessing and gait cycle detection (one trial)

In [ ]:
from gait_analysis.analysis.gait_cycle import (
    build_peak_sequence,
    correct_peak_artifacts,
    detect_peaks,
    extract_phases,
)
from gait_analysis.preprocessing.emg import remove_outliers, replace_nan_with_mean, resample_emg
from gait_analysis.preprocessing.rom import preprocess_rom
from gait_analysis.visualization.emg_plots import plot_emg_coactivation, plot_emg_raw
from gait_analysis.visualization.rom_plots import plot_gait_cycle_normalized, plot_rom_with_phases

key = sample_key
emg_df = emg_data[key]
rom_df = rom_data.get(key)

# Plot raw EMG
raw_signals = {col: emg_df[col].values for col in emg_df.columns}
fig = plot_emg_raw(raw_signals, title=f"Raw EMG — {key}")
plt.show()

In [ ]:
if rom_df is not None and "left_hip" in rom_df.columns:
    raw_rom = rom_df["left_hip"].dropna().values
    proc_rom = preprocess_rom(raw_rom)

    maxima, minima = detect_peaks(proc_rom)
    seq = build_peak_sequence(maxima, minima)
    seq = correct_peak_artifacts(proc_rom, seq)
    phases = extract_phases(proc_rom, seq)

    print(f"Swing phases detected : {len(phases['swing'])}")
    print(f"Stance phases detected: {len(phases['stance'])}")

    fig = plot_rom_with_phases(proc_rom, phases, side="left", title=f"ROM left hip — {key}")
    plt.show()
else:
    print("No matching ROM file for this EMG key.")

In [ ]:
# Gait cycle normalized ROM — both hips at 0-100%
if rom_df is not None and "right_hip" in rom_df.columns:
    right_proc = preprocess_rom(rom_df["right_hip"].dropna().values)
    fig = plot_gait_cycle_normalized(
        proc_rom, right_proc,
        left_phases=phases,
        output_path=None,
    )
    plt.show()

In [ ]:
# EMG coactivation plot — tibialis vs gastrocnemius (left side)
tib = resample_emg(replace_nan_with_mean(remove_outliers(emg_df["rms_left_tibialis"]).values))
gas = resample_emg(replace_nan_with_mean(remove_outliers(emg_df["rms_left_gastrocnemius"]).values))
fig = plot_emg_coactivation(
    tib, gas,
    agonist_label="Tibialis anterior (left)",
    antagonist_label="Gastrocnemius (left)",
    title=f"EMG coactivation — {key}",
)
plt.show()

## Run full pipeline

In [ ]:
from gait_analysis.pipeline import run_pipeline

results = run_pipeline(emg_dir=EMG_DIR, rom_dir=ROM_DIR, output_dir=OUTPUT_DIR)
print(f"Trials processed: {len(results)}")
results.head()

## Statistical visualizations

Replicates the box plots and bar charts from the paper (Figs. 5 and 7).

In [ ]:
import gait_analysis.schema as sc
from gait_analysis.visualization.stats_plots import plot_mad_bar_chart, plot_metric_boxplot

# ROM swing amplitude by BWS level (paper Fig. 5 style)
if sc.ROM_SWING_LEFT in results.columns:
    fig = plot_metric_boxplot(
        results,
        metric=sc.ROM_SWING_LEFT,
        title="Hip ROM during swing phase vs Body Weight Support",
        ylabel="ROM swing left (°)",
    )
    plt.show()

In [ ]:
# MAD bar chart (paper Fig. 7 style)
if sc.MAD_TIBIALIS_LEFT in results.columns and sc.MAD_GASTROCNEMIUS_LEFT in results.columns:
    fig = plot_mad_bar_chart(
        results,
        tibialis_col=sc.MAD_TIBIALIS_LEFT,
        gastrocnemius_col=sc.MAD_GASTROCNEMIUS_LEFT,
        group_by=sc.VELOCITY,
    )
    plt.show()

In [ ]:
# Coactivation index vs body weight support
from gait_analysis.visualization.stats_plots import plot_coactivation_scatter

if sc.CI_LEFT in results.columns:
    fig = plot_coactivation_scatter(
        results,
        ci_col=sc.CI_LEFT,
        x_col=sc.WEIGHT_SUPPORT,
    )
    plt.show()